In [1]:
import sys
sys.path.append("/home/info_core")


from exchange_plugin.okx_plug import InitOkxAdaptor
from exchange_plugin.upbit_plug import InitUpbitAdaptor
from exchange_plugin.binance_plug import InitBinanceAdaptor
from exchange_plugin.bithumb_plug import InitBithumbAdaptor
from exchange_plugin.bybit_plug import InitBybitAdaptor
from etc.db_handler.mongodb_client import InitDBClient # TEMP
from etc.register_monitor_msg import RegisterMonitorMsg # TEMP
import pandas as pd
import json
import time
import datetime
from threading import Thread
import numpy as np

info_dict = {}
info_thread_dict = {}

okx_adaptor = InitOkxAdaptor("fcb66a6e-0443-4743-8d9b-61caf7eab662", "0029E09874B38F5AC7E68E9DFC348667", "121431Rn!!")
upbit_adaptor = InitUpbitAdaptor("x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8", "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL", info_dict, None)
binance_adaptor = InitBinanceAdaptor("4PFfYsObYyUaMlk7m6tT9qIl8X8P3WCUsyu1lEyZ4h50VfTMPIQCNdL9eOJCl3Lb", "011Z1W9p4425AZgtCNbs5L1d77ehZFNmBIwT0pz1SJGP7EiOlPILfWBglbVsxMcK", info_dict, None)
bithumb_adaptor = InitBithumbAdaptor()
bybit_adaptor = InitBybitAdaptor()

# UPBIT SPOT (KRW, BTC Market)
# UPBIT wallet status
# BINANCE SPOT, USD-M Futures, COIN-M Futures
data_name_list = [
        "upbit_spot_info_df",
        "upbit_spot_ticker_df",
        # "upbit_wallet_status_df",
        "binance_spot_ticker_df",
        "binance_spot_info_df",
        "binance_usd_m_ticker_df",
        "binance_usd_m_info_df",
        "binance_coin_m_ticker_df",
        "binance_coin_m_info_df",
        "okx_spot_ticker_df",
        "okx_spot_info_df",
        "okx_usd_m_ticker_df",
        "okx_usd_m_info_df",
        "okx_coin_m_ticker_df",
        "okx_coin_m_info_df",
        "bithumb_spot_info_df",
        "bithumb_spot_ticker_df",
        # "bithumb_wallet_status_df",
        "bybit_spot_info_df",
        "bybit_spot_ticker_df",
        "bybit_usd_m_info_df",
        "bybit_usd_m_ticker_df",
        "bybit_coin_m_info_df",
        "bybit_coin_m_ticker_df"
    ]

def update_exchange_info_as_df(data_name, loop_time_secs=15):
    if data_name == "upbit_spot_ticker_df":
        info_dict[data_name] = upbit_adaptor.spot_all_tickers()
    elif data_name == "upbit_wallet_status_df":
        info_dict[data_name] = upbit_adaptor.wallet_status()
    elif data_name == "upbit_spot_info_df":
        info_dict[data_name] = upbit_adaptor.spot_exchange_info()
    elif data_name == "binance_spot_ticker_df":
        info_dict[data_name] = binance_adaptor.spot_all_tickers()
    elif data_name == "binance_spot_info_df":
        info_dict[data_name] = binance_adaptor.spot_exchange_info()
    elif data_name == "binance_usd_m_ticker_df":
        info_dict[data_name] = binance_adaptor.usd_m_all_tickers()
    elif data_name == "binance_usd_m_info_df":
        info_dict[data_name] = binance_adaptor.usd_m_exchange_info()
    elif data_name == "binance_coin_m_ticker_df":
        info_dict[data_name] = binance_adaptor.coin_m_all_tickers()
    elif data_name == "binance_coin_m_info_df":
        info_dict[data_name] = binance_adaptor.coin_m_exchange_info()
    elif data_name == "okx_spot_ticker_df":
        info_dict[data_name] = okx_adaptor.spot_all_tickers()
    elif data_name == "okx_spot_info_df":
        info_dict[data_name] = okx_adaptor.spot_exchange_info()
    elif data_name == "okx_usd_m_ticker_df":
        info_dict[data_name] = okx_adaptor.usd_m_all_tickers()
    elif data_name == "okx_usd_m_info_df":
        info_dict[data_name] = okx_adaptor.usd_m_exchange_info()
    elif data_name == "okx_coin_m_ticker_df":
        info_dict[data_name] = okx_adaptor.coin_m_all_tickers()
    elif data_name == "okx_coin_m_info_df":
        info_dict[data_name] = okx_adaptor.coin_m_exchange_info()
    elif data_name == "bithumb_spot_info_df":
        info_dict[data_name] = bithumb_adaptor.spot_exchange_info()
    elif data_name == "bithumb_spot_ticker_df":
        info_dict[data_name] = bithumb_adaptor.spot_all_tickers()
    elif data_name == "bithumb_wallet_status_df":
        info_dict[data_name] = bithumb_adaptor.wallet_status()
    elif data_name == "bybit_spot_info_df":
        info_dict[data_name] = bybit_adaptor.spot_exchange_info()
    elif data_name == "bybit_spot_ticker_df":
        info_dict[data_name] = bybit_adaptor.spot_all_tickers()
    elif data_name == "bybit_usd_m_info_df":
        info_dict[data_name] = bybit_adaptor.usd_m_exchange_info()
    elif data_name == "bybit_usd_m_ticker_df":
        info_dict[data_name] = bybit_adaptor.usd_m_all_tickers()
    elif data_name == "bybit_coin_m_info_df":
        info_dict[data_name] = bybit_adaptor.coin_m_exchange_info()
    elif data_name == "bybit_coin_m_ticker_df":
        info_dict[data_name] = bybit_adaptor.coin_m_all_tickers()
    else:
        print(f"update_exchange_info_as_df|name:{data_name} is not valid.")

for data_name in data_name_list:
    info_thread_dict[f"update_{data_name}"] = Thread(target=update_exchange_info_as_df, args=(data_name,), daemon=True)
    info_thread_dict[f"update_{data_name}"].start()
    print(f"InitCore|update_{data_name} thread has started.")


with open("/home/info_core/info_core_config.json") as f:
        config = json.load(f)

node = config['node']
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id_list = []
admin_id = config['telegram_admin_id']['charlie1155']
admin_id_list.append(admin_id)
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir=None)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

enabled_market_klines = config['node_settings'][node]['enabled_market_klines']
# total enabled market settings
total_enabled_market_klines = []
for each_market_code_list in config['node_settings'].values():
    total_enabled_market_klines += each_market_code_list['enabled_market_klines']

2023-11-24 13:20:59,789 - okx_plug - INFO - okx_plug_logger started.
2023-11-24 13:21:00,176 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-24 13:21:00,294 - binance_plug - INFO - binance_plug_logger started.
2023-11-24 13:21:00,295 - bithumb_plug - INFO - bithumb_plug_logger started.
2023-11-24 13:21:00,297 - bybit_plug - INFO - bybit_plug_logger started.


InitCore|update_upbit_spot_info_df thread has started.
InitCore|update_upbit_spot_ticker_df thread has started.
InitCore|update_binance_spot_ticker_df thread has started.
InitCore|update_binance_spot_info_df thread has started.
InitCore|update_binance_usd_m_ticker_df thread has started.
InitCore|update_binance_usd_m_info_df thread has started.
InitCore|update_binance_coin_m_ticker_df thread has started.
InitCore|update_binance_coin_m_info_df thread has started.
InitCore|update_okx_spot_ticker_df thread has started.
InitCore|update_okx_spot_info_df thread has started.
InitCore|update_okx_usd_m_ticker_df thread has started.
InitCore|update_okx_usd_m_info_df thread has started.
InitCore|update_okx_coin_m_ticker_df thread has started.
InitCore|update_okx_coin_m_info_df thread has started.
InitCore|update_bithumb_spot_info_df thread has started.
InitCore|update_bithumb_spot_ticker_df thread has started.
InitCore|update_bybit_spot_info_df thread has started.
InitCore|update_bybit_spot_ticker

2023-11-24 13:21:00,861 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_coin_m_info_df') is None, Fetched from API
2023-11-24 13:21:01,987 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_spot_info_df') is None, Fetched from API
2023-11-24 13:21:02,044 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_usd_m_info_df') is None, Fetched from API
2023-11-24 13:21:02,246 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_usd_m_info_df') is None, Fetched from API
2023-11-24 13:21:03,006 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_spot_ticker_df') is None, Fetched from API


In [2]:
exchange_list = []
for each_market_code_combi in total_enabled_market_klines:
    first_market_code, second_market_code = each_market_code_combi.split(":")
    first_market, first_quote_asset = first_market_code.split('/')
    first_exchange = first_market.split('_')[0]
    first_market_type = first_market.replace(f'{first_exchange}_','')
    second_market, second_quote_asset = second_market_code.split('/')
    second_exchange = second_market_code.split('_')[0]
    second_market_type = second_market.replace(f'{second_exchange}_','')
    if first_market_type in ["USD_M", "COIN_M"]:
        exchange_list.append(first_exchange)
    if second_market_type in ["USD_M", "COIN_M"]:
        exchange_list.append(second_exchange)
exchange_list = list(set(exchange_list))
perpetual_tup_list = []
for each_exchange in exchange_list:
    for each_market_type in ['USD_M', 'COIN_M']:
        perpetual_tup_list.append((each_exchange, each_market_type))

perpetual_combination_list = []
for i, perpetual_tup_one in enumerate(perpetual_tup_list):
    for perpetual_tup_two in perpetual_tup_list[i+1:]:
        perpetual_combination_list.append((perpetual_tup_one, perpetual_tup_two))

perpetual_combination_list

[(('BINANCE', 'USD_M'), ('BINANCE', 'COIN_M')),
 (('BINANCE', 'USD_M'), ('OKX', 'USD_M')),
 (('BINANCE', 'USD_M'), ('OKX', 'COIN_M')),
 (('BINANCE', 'USD_M'), ('BYBIT', 'USD_M')),
 (('BINANCE', 'USD_M'), ('BYBIT', 'COIN_M')),
 (('BINANCE', 'COIN_M'), ('OKX', 'USD_M')),
 (('BINANCE', 'COIN_M'), ('OKX', 'COIN_M')),
 (('BINANCE', 'COIN_M'), ('BYBIT', 'USD_M')),
 (('BINANCE', 'COIN_M'), ('BYBIT', 'COIN_M')),
 (('OKX', 'USD_M'), ('OKX', 'COIN_M')),
 (('OKX', 'USD_M'), ('BYBIT', 'USD_M')),
 (('OKX', 'USD_M'), ('BYBIT', 'COIN_M')),
 (('OKX', 'COIN_M'), ('BYBIT', 'USD_M')),
 (('OKX', 'COIN_M'), ('BYBIT', 'COIN_M')),
 (('BYBIT', 'USD_M'), ('BYBIT', 'COIN_M'))]

In [5]:
temp = info_dict['binance_usd_m_info_df']
temp[temp['perpetual']==False]

,symbol,pair,contractType,deliveryDate,onboardDate,status,maintMarginPercent,requiredMarginPercent,base_asset,quote_asset,...,underlyingSubType,settlePlan,triggerProtect,liquidationFee,marketTakeBound,maxMoveOrderLimit,filters,orderTypes,timeInForce,perpetual
89,BTCSTUSDT,BTCSTUSDT,NaN,4133404800000,1614754800000,PENDING_TRADING,2.5,5.0,BTCST,USDT,...,[DEFI],0,0.15,0.04,0.30,10000,"[{'tickSize': '0.001', 'filterType': 'PRICE_FI...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",False
246,BTCUSDT_231229,BTCUSDT,CURRENT_QUARTER,1703836800000,1692360000000,TRADING,2.5,5.0,BTC,USDT,...,[],0,0.05,0.01,0.05,10000,"[{'tickSize': '0.1', 'minPrice': '576.3', 'max...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",False
247,ETHUSDT_231229,ETHUSDT,CURRENT_QUARTER,1703836800000,1692360900000,TRADING,2.5,5.0,ETH,USDT,...,[],0,0.05,0.01,0.05,10000,"[{'minPrice': '41.10', 'maxPrice': '100000', '...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",False
254,BTCUSDT_240329,BTCUSDT,NEXT_QUARTER,1711699200000,1695974400000,TRADING,2.5,5.0,BTC,USDT,...,[],0,0.05,0.01,0.05,10000,"[{'minPrice': '576.3', 'maxPrice': '1000000', ...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",False
255,ETHUSDT_240329,ETHUSDT,NEXT_QUARTER,1711699200000,1695974400000,TRADING,2.5,5.0,ETH,USDT,...,[],0,0.05,0.01,0.05,10000,"[{'maxPrice': '100000', 'tickSize': '0.01', 'f...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",False


In [6]:
temp = info_dict['binance_coin_m_info_df']
temp[temp['perpetual']==False]

,symbol,pair,contractType,deliveryDate,onboardDate,contractStatus,contractSize,marginAsset,maintMarginPercent,requiredMarginPercent,...,maxMoveOrderLimit,triggerProtect,underlyingType,underlyingSubType,filters,orderTypes,timeInForce,liquidationFee,marketTakeBound,perpetual
1,BTCUSD_231229,BTCUSD,CURRENT_QUARTER,1703836800000,1688112000000,TRADING,100,BTC,2.5,5.0,...,10000,0.05,COIN,[],"[{'minPrice': '1000', 'maxPrice': '4671848', '...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False
2,BTCUSD_240329,BTCUSD,NEXT_QUARTER,1711699200000,1695974400000,TRADING,100,BTC,2.5,5.0,...,10000,0.05,COIN,[],"[{'minPrice': '1000', 'maxPrice': '4671848', '...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False
4,ETHUSD_231229,ETHUSD,CURRENT_QUARTER,1703836800000,1688112000000,TRADING,10,ETH,2.5,5.0,...,10000,0.05,COIN,[],"[{'minPrice': '50', 'maxPrice': '316265', 'fil...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False
5,ETHUSD_240329,ETHUSD,NEXT_QUARTER,1711699200000,1695974400000,TRADING,10,ETH,2.5,5.0,...,10000,0.05,COIN,[],"[{'minPrice': '50', 'maxPrice': '316265', 'fil...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False
27,GALAUSD_PERP,GALAUSD,PERPETUAL DELIVERING,1672045200000,1638342000000,DELIVERING,10,GALA,2.5,5.0,...,10000,0.05,COIN,[Metaverse],"[{'minPrice': '0.00001', 'maxPrice': '5', 'fil...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.025,0.05,False
39,VETUSD_PERP,VETUSD,PERPETUAL DELIVERING,1672045200000,1648537200000,DELIVERING,10,VET,2.5,5.0,...,10000,0.05,COIN,[Layer-1],"[{'minPrice': '0.000010', 'maxPrice': '10000',...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False
40,ZILUSD_PERP,ZILUSD,PERPETUAL DELIVERING,1672045200000,1649228400000,DELIVERING,10,ZIL,2.5,5.0,...,10000,0.12,COIN,[Layer-1],"[{'minPrice': '0.000010', 'maxPrice': '10000',...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.020,0.12,False
48,ADAUSD_231229,ADAUSD,CURRENT_QUARTER,1703836800000,1688112000000,TRADING,10,ADA,2.5,5.0,...,10000,0.05,COIN,[],"[{'minPrice': '0.01700', 'maxPrice': '205', 'f...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False
49,LINKUSD_231229,LINKUSD,CURRENT_QUARTER,1703836800000,1688112000000,TRADING,10,LINK,2.5,5.0,...,10000,0.05,COIN,[],"[{'minPrice': '0.500', 'maxPrice': '2737', 'fi...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False
50,BCHUSD_231229,BCHUSD,CURRENT_QUARTER,1703836800000,1688112000000,TRADING,10,BCH,2.5,5.0,...,10000,0.05,COIN,[],"[{'minPrice': '14.40', 'maxPrice': '66208', 'f...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX]",0.010,0.05,False


In [37]:

futures_info_df = info_dict['binance_coin_m_info_df']
futures_symbol_list = futures_info_df[futures_info_df['perpetual']==False]['symbol'].to_list()
futures_ticker_df = info_dict['binance_coin_m_ticker_df']
futures_ticker_df = futures_ticker_df[futures_ticker_df['symbol'].isin(futures_symbol_list)][['symbol','lastPrice','volume','baseVolume','base_asset','quote_asset','atp24h']]

perp_info_df = info_dict['binance_usd_m_info_df']
perp_symbol_list = perp_info_df[perp_info_df['perpetual']==True]['symbol'].to_list()
perp_ticker_df = info_dict['binance_usd_m_ticker_df']
perp_ticker_df = perp_ticker_df[perp_ticker_df['symbol'].isin(perp_symbol_list)][['symbol','lastPrice','base_asset','quote_asset','atp24h']]
merged_ticker_df = futures_ticker_df.merge(perp_ticker_df, on=['base_asset'])
merged_ticker_df['basis_pct'] = ((merged_ticker_df['lastPrice_x'] - merged_ticker_df['lastPrice_y']) / merged_ticker_df['lastPrice_y'] * 100).round(3)
merged_ticker_df['left_days'] = merged_ticker_df['symbol_x'].apply(lambda x: (datetime.datetime.strptime(x.split('_')[1], "%y%m%d") - datetime.datetime.utcnow()).days)
merged_ticker_df

,symbol_x,lastPrice_x,volume,baseVolume,base_asset,quote_asset_x,atp24h_x,symbol_y,lastPrice_y,quote_asset_y,atp24h_y,basis_pct,left_days
0,BNBUSD_240329,235.0700,123318,5.258466e+03,BNB,USD,12331800,BNBBUSD,235.55000,BUSD,8.607017e+06,-0.204,125
1,BNBUSD_240329,235.0700,123318,5.258466e+03,BNB,USD,12331800,BNBUSDT,235.55000,USDT,3.320071e+08,-0.204,125
2,BNBUSD_231229,234.4600,138402,5.921373e+03,BNB,USD,13840200,BNBBUSD,235.55000,BUSD,8.607017e+06,-0.463,34
3,BNBUSD_231229,234.4600,138402,5.921373e+03,BNB,USD,13840200,BNBUSDT,235.55000,USDT,3.320071e+08,-0.463,34
4,ADAUSD_240329,0.4010,36507,9.307358e+05,ADA,USD,3650700,ADABUSD,0.27640,BUSD,7.186425e+06,45.080,125
5,ADAUSD_240329,0.4010,36507,9.307358e+05,ADA,USD,3650700,ADAUSDT,0.39390,USDT,1.860769e+08,1.802,125
6,ADAUSD_231229,0.3960,60041,1.548920e+06,ADA,USD,6004100,ADABUSD,0.27640,BUSD,7.186425e+06,43.271,34
7,ADAUSD_231229,0.3960,60041,1.548920e+06,ADA,USD,6004100,ADAUSDT,0.39390,USDT,1.860769e+08,0.533,34
8,LINKUSD_240329,14.7530,123165,8.293958e+04,LINK,USD,12316500,LINKBUSD,6.54700,BUSD,4.334334e+06,125.340,125
9,LINKUSD_240329,14.7530,123165,8.293958e+04,LINK,USD,12316500,LINKUSDT,14.36000,USDT,4.059425e+08,2.737,125
